# 01_decoding — Hue classification (LORO) and hue interpolation (LOCO)

**Manuscript:** Results section 1; Figure 4; Methods 'Hue-channel basis model', 'Two decoding schemes'; Supplementary S5 (decoders), S6 (GCV), S7 (cross-validation), S16 (effect sizes), S17 (alignment robustness).

Leave-one-run-out (LORO) classification and leave-one-color-out (LOCO) interpolation with the hue-channel basis model (six half-wave-rectified squared-cosine channels). `loro_baseline.py` / `loco_baseline.py` run the six-decoder comparison of S5 on the SRM-aligned amplitudes; the main-text readouts use the Procrustes-aligned amplitudes through `common/loco_canonical.py`. `perm_adjacent_n7.py` is the 1,000-permutation colour-label null for the control interpolation gate. `validation_tests.py` holds the cross-participant encoder transfer (Mann-Whitney).

**How to read this notebook.** Every code cell loads committed result files from `results/` and compares the values it derives with the numbers printed in the manuscript (`V.check`). A check passes when the produced value equals the printed one at the printed precision, or satisfies the stated relation. Quantities that have no committed artifact are recorded as pointers (`V.flag`) rather than silently omitted. The last cell tallies the checks and writes `_checks_01_decoding.json`, which `run_notebooks.py` collects into `REPORT.md`.

Provenance: built by `tools/public_repo/build.py` of the development repository (commit 53c81c2); manuscript source in `../paper/`; check list in `../MANIFEST.md`; code map in `../MAP.md`.

**Source and code map**

| Result file | Producing script | What it holds |
|---|---|---|
| `results/loro/procrustes/sub-*_performance_raw.json` | `scripts/loro_baseline.py` | LORO eight-way accuracy per fold, Procrustes-aligned amplitudes (main text, tab:alignment) |
| `results/loro/srm/sub-*_performance_raw.json` | `scripts/loro_baseline.py` | LORO per fold, SRM-aligned amplitudes, six decoders (tab:loro_decoders) |
| `results/loco/srm/sub-*_loco.json` | `scripts/loco_baseline.py` | LOCO adjacent accuracy, SRM-aligned amplitudes, six decoders (tab:loco_decoders) |
| `results/adjacent_accuracy_per_hue.json` | `common/loco_canonical.py via tools export` | per-hue LOCO adjacent / exact accuracy, Procrustes space, both pipelines (Results section 1, Figure 4B-C, tab:effect_sizes, tab:alignment) |
| `results/perm_adjacent_n7.json, perm_n7_null_*.npy` | `scripts/perm_adjacent_n7.py` | control interpolation gate: 1,000 colour-label permutations, n = 7 (tab:interp_arms, primary) |
| `results/cross_subject_generalization.json` | `scripts/validation_tests.py` | control-trained encoder applied to held-out controls (28 cells) and to CVD participants (12 cells incl. sub-10) |
| `results/nested_procrustes/nested_only/` | `scripts/loro_baseline.py (nested variant)` | leakage bound of the fixed-reference alignment (S7); ten participants incl. sub-10 |
| `results/lambda_stability.json` | `scripts/lambda_stability_loco.py` | ridge-penalty grid (S6) |

In [1]:
import sys, json, csv
from pathlib import Path
sys.path.insert(0, str((Path.cwd() / ".." / "common").resolve()))
import numpy as np
from scipy import stats
import verify as V
from stats_helpers import crawford_howell, hedges_g, bh_fdr, wilson_interval
R = Path("results")
def J(name):
    with open(R / name) as f:
        return json.load(f)
HC = [f"sub-{i:02d}" for i in range(1, 8)]
CVD = {"deutan": "sub-08", "protan": "sub-09"}
ROIS = ["V1", "V2", "V3", "hV4"]
HUES = ["red", "orange", "yellow", "green", "cyan", "blue", "purple", "magenta"]

SUBS = HC + ["sub-08", "sub-09"]
DEC = ["LDA", "SVM", "ForwardEncoding", "Ridge", "KernelRidge", "MLP"]
DEC_NAME = {"LDA": "LDA", "SVM": "SVM", "ForwardEncoding": "Hue-channel basis", "Ridge": "Ridge", "KernelRidge": "Kernel ridge", "MLP": "MLP"}
RDIR = {"V1": "V1", "V2": "V2", "V3": "V3", "hV4": "V4"}
def loro(space, sub, roi, model="ForwardEncoding"):
    folds = J(f"loro/{space}/{sub}_performance_raw.json")["results"][space][RDIR[roi]][model]
    return float(np.mean([f["acc_exact"] for f in folds]))
def loco_srm(sub, roi, model="ForwardEncoding"):
    return float(J(f"loco/srm/{sub}_loco.json")["results"][RDIR[roi]][model]["overall_adjacent_acc"])
adj = J("adjacent_accuracy_per_hue.json")["pipelines"]["primary"]
perm = J("perm_adjacent_n7.json")
xs = J("cross_subject_generalization.json")["ForwardEncoding"]
LORO_P = {sp: {s: {r: loro(sp, s, r) for r in ROIS} for s in SUBS} for sp in ("procrustes", "srm")}

V.start("01_decoding")

### Eight-way classification, Procrustes space (Results section 1 ¶1; Supplementary tab:alignment)
Leave-one-run-out accuracy of the hue-channel basis model, mean over six folds. Chance = 0.125.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 01.01 | tab:alignment | LORO classification, Procrustes space: 4 ROIs x (controls, deutan, protan) | `12 cells, see 01.T1.*` |
| 01.02 | Results §1 ¶1 | lowest CVD classification cell (hV4) | `0.375` |
| 01.03 | Results §1 ¶1 | lowest cell as a multiple of chance | `3.0` |
| 01.04 | Results §1 ¶1 | every CVD cell above chance 0.125 | `0.125` |

In [2]:
REP = {"V1": (0.580, 0.562, 0.562), "V2": (0.607, 0.521, 0.562), "V3": (0.574, 0.396, 0.458), "hV4": (0.488, 0.375, 0.375)}
for roi, (hc_r, d_r, p_r) in REP.items():
    hc = np.mean([LORO_P["procrustes"][s][roi] for s in HC])
    V.check(f"01.T1.{roi}.controls", f"tab:alignment LORO controls {roi}", hc, hc_r, nd=3)
    V.check(f"01.T1.{roi}.deutan", f"tab:alignment LORO deutan {roi}", LORO_P["procrustes"]["sub-08"][roi], d_r, nd=3)
    V.check(f"01.T1.{roi}.protan", f"tab:alignment LORO protan {roi}", LORO_P["procrustes"]["sub-09"][roi], p_r, nd=3)
lowest_cvd = min(LORO_P["procrustes"][s][r] for s in ("sub-08", "sub-09") for r in ROIS)
V.table('01.01', 'tab:alignment | LORO classification, Procrustes space: 4 ROIs x (controls, deutan, protan)', '12 cells, see 01.T1.*')
V.check('01.02', 'Results §1 ¶1 | lowest CVD classification cell (hV4)', lowest_cvd, 0.375, nd=3)
V.check('01.03', 'Results §1 ¶1 | lowest cell as a multiple of chance', lowest_cvd / 0.125, 3.0, nd=1)
V.check('01.04', 'Results §1 ¶1 | every CVD cell above chance 0.125', lowest_cvd, 0.125, mode='gt')

[OK ] 01.T1.V1.controls tab:alignment LORO controls V1: produced=0.5804  reported=0.58
[OK ] 01.T1.V1.deutan tab:alignment LORO deutan V1: produced=0.5625  reported=0.562
[OK ] 01.T1.V1.protan tab:alignment LORO protan V1: produced=0.5625  reported=0.562
[OK ] 01.T1.V2.controls tab:alignment LORO controls V2: produced=0.6071  reported=0.607
[OK ] 01.T1.V2.deutan tab:alignment LORO deutan V2: produced=0.5208  reported=0.521
[OK ] 01.T1.V2.protan tab:alignment LORO protan V2: produced=0.5625  reported=0.562
[OK ] 01.T1.V3.controls tab:alignment LORO controls V3: produced=0.5744  reported=0.574
[OK ] 01.T1.V3.deutan tab:alignment LORO deutan V3: produced=0.3958  reported=0.396
[OK ] 01.T1.V3.protan tab:alignment LORO protan V3: produced=0.4583  reported=0.458
[OK ] 01.T1.hV4.controls tab:alignment LORO controls hV4: produced=0.4881  reported=0.488
[OK ] 01.T1.hV4.deutan tab:alignment LORO deutan hV4: produced=0.375  reported=0.375
[OK ] 01.T1.hV4.protan tab:alignment LORO protan hV4: prod

### Single-case classification contrasts (Supplementary S16)
Crawford-Howell modified t against the seven controls, two-tailed because the hypothesis is preservation.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 01.05 | S16 | smallest |d_cc| over the eight classification contrasts | `0.25` |
| 01.06 | S16 | largest |d_cc| | `1.58` |
| 01.07 | S16 | smallest two-tailed p | `0.189` |
| 01.08 | S16 | deutan V3 d_cc | `-1.58` |
| 01.09 | S16 | hV4 d_cc, both participants | `-1.08` |
| 01.10 | S16 | hV4 two-tailed p, both participants | `0.352` |

In [3]:
ch_loro = {}
for s in ("sub-08", "sub-09"):
    for roi in ROIS:
        hc = [LORO_P["procrustes"][h][roi] for h in HC]
        t, p, d = crawford_howell(LORO_P["procrustes"][s][roi], hc, tail="two")
        ch_loro[(s, roi)] = (t, p, d)
        print(f"{s} {roi}: t={t:+.2f} p={p:.3f} d_cc={d:+.2f}")
absd = [abs(v[2]) for v in ch_loro.values()]; ps = [v[1] for v in ch_loro.values()]
V.check('01.05', 'S16 | smallest |d_cc| over the eight classification contrasts', min(absd), 0.25, nd=2)
V.check('01.06', 'S16 | largest |d_cc|', max(absd), 1.58, nd=2)
V.check('01.07', 'S16 | smallest two-tailed p', min(ps), 0.189, nd=3)
V.check('01.08', 'S16 | deutan V3 d_cc', ch_loro[("sub-08", "V3")][2], -1.58, nd=2)
V.check('01.09', 'S16 | hV4 d_cc, both participants', ch_loro[("sub-08", "hV4")][2], -1.08, nd=2)
V.check('01.10', 'S16 | hV4 two-tailed p, both participants', ch_loro[("sub-09", "hV4")][1], 0.352, nd=3)

sub-08 V1: t=-0.24 p=0.821 d_cc=-0.25
sub-08 V2: t=-0.71 p=0.502 d_cc=-0.76
sub-08 V3: t=-1.48 p=0.189 d_cc=-1.58
sub-08 hV4: t=-1.01 p=0.352 d_cc=-1.08
sub-09 V1: t=-0.24 p=0.821 d_cc=-0.25
sub-09 V2: t=-0.37 p=0.725 d_cc=-0.39
sub-09 V3: t=-0.96 p=0.373 d_cc=-1.03
sub-09 hV4: t=-1.01 p=0.352 d_cc=-1.08
[OK ] 01.05 S16 | smallest |d_cc| over the eight classification contrasts: produced=0.253  reported=0.25
[OK ] 01.06 S16 | largest |d_cc|: produced=1.584  reported=1.58
[OK ] 01.07 S16 | smallest two-tailed p: produced=0.189  reported=0.189
[OK ] 01.08 S16 | deutan V3 d_cc: produced=-1.584  reported=-1.58
[OK ] 01.09 S16 | hV4 d_cc, both participants: produced=-1.08  reported=-1.08
[OK ] 01.10 S16 | hV4 two-tailed p, both participants: produced=0.3515  reported=0.352


### Control-trained encoder applied to CVD cortex (Results section 1 ¶2; Supplementary S5 last paragraph)
The committed file stores 12 HC-to-CVD cells (three CVD participants x four ROIs, ROI-major order). sub-10 is excluded from the manuscript, so the notebook keeps the eight sub-08 / sub-09 cells.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 01.11 | Results §1 ¶2 | HC-to-CVD mean accuracy over eight cells | `0.432` |
| 01.12 | Results §1 ¶2 | number of CVD cells | `8` |
| 01.13 | Results §1 ¶2 | t(7) against chance 0.125 | `6.51` |
| 01.14 | Results §1 ¶2 | p against chance (< 0.001) | `0.001` |
| 01.15 | Results §1 ¶2 | HC-to-HC mean over 28 cells | `0.526` |
| 01.16 | S5 | Mann-Whitney U | `163.5` |
| 01.17 | Results §1 ¶2 | Mann-Whitney p | `0.052` |
| 01.18 | S5 | rank-biserial r | `0.46` |
| 01.19 | S5 | lowest single HC-to-CVD cell | `0.271` |

In [4]:
hh = np.array(xs["hc_to_hc"]["scores"])
hcvd_all = np.array(xs["hc_to_cvd"]["scores"])
hcvd = hcvd_all[[i for i in range(len(hcvd_all)) if i % 3 != 2]]   # drop sub-10 (third participant in each ROI block)
t1, p1 = stats.ttest_1samp(hcvd, 0.125)
U, pU = stats.mannwhitneyu(hh, hcvd, alternative="two-sided")
rb = 1 - 2 * U / (len(hh) * len(hcvd))
print(f"HC->HC {hh.mean():.3f} (n={len(hh)}), HC->CVD {hcvd.mean():.3f} (n={len(hcvd)}), U={U}, p={pU:.3f}, r_rb={rb:+.2f}")
V.check('01.11', 'Results §1 ¶2 | HC-to-CVD mean accuracy over eight cells', hcvd.mean(), 0.432, nd=3)
V.check('01.12', 'Results §1 ¶2 | number of CVD cells', len(hcvd), 8, mode='eq')
V.check('01.13', 'Results §1 ¶2 | t(7) against chance 0.125', t1, 6.51, nd=2)
V.check('01.14', 'Results §1 ¶2 | p against chance (< 0.001)', p1, 0.001, mode='lt')
V.check('01.15', 'Results §1 ¶2 | HC-to-HC mean over 28 cells', hh.mean(), 0.526, nd=3)
V.check('01.16', 'S5 | Mann-Whitney U', U, 163.5, nd=1)
V.check('01.17', 'Results §1 ¶2 | Mann-Whitney p', pU, 0.052, nd=3)
V.check('01.18', 'S5 | rank-biserial r', abs(rb), 0.46, nd=2)
V.check('01.19', 'S5 | lowest single HC-to-CVD cell', hcvd.min(), 0.271, nd=3)

HC->HC 0.526 (n=28), HC->CVD 0.432 (n=8), U=163.5, p=0.052, r_rb=-0.46
[OK ] 01.11 Results §1 ¶2 | HC-to-CVD mean accuracy over eight cells: produced=0.4323  reported=0.432
[OK ] 01.12 Results §1 ¶2 | number of CVD cells: produced=8  reported=8
[OK ] 01.13 Results §1 ¶2 | t(7) against chance 0.125: produced=6.51  reported=6.51
[OK ] 01.14 Results §1 ¶2 | p against chance (< 0.001): produced=0.0003311  reported=0.001
[OK ] 01.15 Results §1 ¶2 | HC-to-HC mean over 28 cells: produced=0.526  reported=0.526
[OK ] 01.16 S5 | Mann-Whitney U: produced=163.5  reported=163.5
[OK ] 01.17 Results §1 ¶2 | Mann-Whitney p: produced=0.05176  reported=0.052
[OK ] 01.18 S5 | rank-biserial r: produced=0.4598  reported=0.46
[OK ] 01.19 S5 | lowest single HC-to-CVD cell: produced=0.2708  reported=0.271


### Control interpolation gate (Results section 1 ¶3; Supplementary tab:interp_arms, primary column)
LOCO adjacent accuracy of the controls (n = 7) against 1,000 per-participant colour-label permutations. Analytic chance = 0.25 (91 of 360 draws).

| id | manuscript | quantity | reported |
|---|---|---|---|
| 01.20 | Results §1 ¶3 | hV4 control adjacent accuracy | `0.456` |
| 01.21 | Results §1 ¶3 | hV4 SEM (n = 7) | `0.039` |
| 01.22 | Results §1 ¶3 | hV4 permutation p | `0.011` |
| 01.23 | Results §1 ¶3 | V1 control adjacent accuracy | `0.393` |
| 01.24 | Results §1 ¶3 | V1 permutation p | `0.164` |
| 01.25 | Results §1 ¶3 | V2 control adjacent accuracy | `0.357` |
| 01.26 | Results §1 ¶3 | V2 permutation p | `0.424` |
| 01.27 | Results §1 ¶3 | V3 control adjacent accuracy | `0.339` |
| 01.28 | Results §1 ¶3 | V3 permutation p | `0.586` |
| 01.29 | Results §1 ¶3 | permutation null lies near 0.35 (all four ROIs within 0.34-0.36) | `(0.34, 0.36)` |
| 01.30 | Results §1 ¶3 | all four ROIs exceed analytic chance 0.25 | `0.25` |
| 01.31 | Results §1 ¶3 | number of permutations | `1000` |

In [5]:
for roi in ROIS:
    null = np.load(R / f"perm_n7_null_{roi}.npy")
    print(f"{roi}: observed {perm[roi]['observed']:.3f}  p={perm[roi]['p_perm']:.3f}  null mean {null.mean():.3f} (stored {perm[roi]['null_mean']:.3f})")
null_means = [perm[r]["null_mean"] for r in ROIS]
V.check('01.20', 'Results §1 ¶3 | hV4 control adjacent accuracy', perm["hV4"]["observed"], 0.456, nd=3)
V.check('01.21', 'Results §1 ¶3 | hV4 SEM (n = 7)', perm["hV4"]["hc_sem"], 0.039, nd=3)
V.check('01.22', 'Results §1 ¶3 | hV4 permutation p', perm["hV4"]["p_perm"], 0.011, nd=3)
V.check('01.23', 'Results §1 ¶3 | V1 control adjacent accuracy', perm["V1"]["observed"], 0.393, nd=3)
V.check('01.24', 'Results §1 ¶3 | V1 permutation p', perm["V1"]["p_perm"], 0.164, nd=3)
V.check('01.25', 'Results §1 ¶3 | V2 control adjacent accuracy', perm["V2"]["observed"], 0.357, nd=3)
V.check('01.26', 'Results §1 ¶3 | V2 permutation p', perm["V2"]["p_perm"], 0.424, nd=3)
V.check('01.27', 'Results §1 ¶3 | V3 control adjacent accuracy', perm["V3"]["observed"], 0.339, nd=3)
V.check('01.28', 'Results §1 ¶3 | V3 permutation p', perm["V3"]["p_perm"], 0.586, nd=3)
V.check('01.29', 'Results §1 ¶3 | permutation null lies near 0.35 (all four ROIs within 0.34-0.36)', (min(null_means), max(null_means)), (0.34, 0.36), mode='range')
V.check('01.30', 'Results §1 ¶3 | all four ROIs exceed analytic chance 0.25', min(perm[r]["observed"] for r in ROIS), 0.25, mode='gt')
V.check('01.31', 'Results §1 ¶3 | number of permutations', perm["hV4"]["n_perms"], 1000, mode='eq')

V1: observed 0.393  p=0.164  null mean 0.346 (stored 0.346)
V2: observed 0.357  p=0.424  null mean 0.349 (stored 0.349)
V3: observed 0.339  p=0.586  null mean 0.347 (stored 0.347)
hV4: observed 0.456  p=0.011  null mean 0.346 (stored 0.346)
[OK ] 01.20 Results §1 ¶3 | hV4 control adjacent accuracy: produced=0.456  reported=0.456
[OK ] 01.21 Results §1 ¶3 | hV4 SEM (n = 7): produced=0.03854  reported=0.039
[OK ] 01.22 Results §1 ¶3 | hV4 permutation p: produced=0.01099  reported=0.011
[OK ] 01.23 Results §1 ¶3 | V1 control adjacent accuracy: produced=0.3929  reported=0.393
[OK ] 01.24 Results §1 ¶3 | V1 permutation p: produced=0.1638  reported=0.164
[OK ] 01.25 Results §1 ¶3 | V2 control adjacent accuracy: produced=0.3571  reported=0.357
[OK ] 01.26 Results §1 ¶3 | V2 permutation p: produced=0.4236  reported=0.424
[OK ] 01.27 Results §1 ¶3 | V3 control adjacent accuracy: produced=0.3393  reported=0.339
[OK ] 01.28 Results §1 ¶3 | V3 permutation p: produced=0.5864  reported=0.586
[OK ] 0

### Single-case interpolation at hV4 (Results section 1 ¶4; Supplementary tab:effect_sizes)
One-tailed (lower) Crawford-Howell against the seven controls, on the per-hue adjacent accuracy averaged over hues.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 01.32 | Results §1 ¶4 | deutan hV4 adjacent accuracy | `0.25` |
| 01.33 | Results §1 ¶4 | deutan Crawford-Howell t | `-1.89` |
| 01.34 | Results §1 ¶4 | deutan p (one-tailed) | `0.054` |
| 01.35 | Results §1 ¶4 | deutan d_cc | `-2.02` |
| 01.36 | Results §1 ¶4 | protan hV4 adjacent accuracy | `0.13` |
| 01.37 | Results §1 ¶4 | protan Crawford-Howell t | `-3.04` |
| 01.38 | Results §1 ¶4 | protan p (one-tailed) | `0.012` |
| 01.39 | Results §1 ¶4 | protan d_cc | `-3.25` |
| 01.40 | Results §1 ¶4 | control mean | `0.46` |
| 01.41 | Results §1 ¶4 | all seven controls above the protan participant | `7` |
| 01.42 | Results §1 ¶4 | at least five controls above the deutan participant | `5` |

In [6]:
hcm = np.array([adj[s]["hV4"]["adjacent_mean"] for s in HC])
d_all = adj["sub-08"]["hV4"]["adjacent_mean"]; p_all = adj["sub-09"]["hV4"]["adjacent_mean"]
t8, pv8, d8 = crawford_howell(d_all, hcm, tail="lower")
t9, pv9, d9 = crawford_howell(p_all, hcm, tail="lower")
print(f"deutan {d_all:.3f}: t={t8:.2f} p={pv8:.3f} d={d8:.2f} | protan {p_all:.3f}: t={t9:.2f} p={pv9:.3f} d={d9:.2f} | controls {hcm.mean():.3f}")
V.check('01.32', 'Results §1 ¶4 | deutan hV4 adjacent accuracy', d_all, 0.25, nd=2)
V.check('01.33', 'Results §1 ¶4 | deutan Crawford-Howell t', t8, -1.89, nd=2)
V.check('01.34', 'Results §1 ¶4 | deutan p (one-tailed)', pv8, 0.054, nd=3)
V.check('01.35', 'Results §1 ¶4 | deutan d_cc', d8, -2.02, nd=2)
V.check('01.36', 'Results §1 ¶4 | protan hV4 adjacent accuracy', p_all, 0.13, nd=2)
V.check('01.37', 'Results §1 ¶4 | protan Crawford-Howell t', t9, -3.04, nd=2)
V.check('01.38', 'Results §1 ¶4 | protan p (one-tailed)', pv9, 0.012, nd=3)
V.check('01.39', 'Results §1 ¶4 | protan d_cc', d9, -3.25, nd=2)
V.check('01.40', 'Results §1 ¶4 | control mean', hcm.mean(), 0.46, nd=2)
V.check('01.41', 'Results §1 ¶4 | all seven controls above the protan participant', int((hcm > p_all).sum()), 7, mode='eq')
V.check('01.42', 'Results §1 ¶4 | at least five controls above the deutan participant', int((hcm > d_all).sum()), 5, mode='ge')

deutan 0.250: t=-1.89 p=0.054 d=-2.02 | protan 0.125: t=-3.04 p=0.011 d=-3.25 | controls 0.456
[OK ] 01.32 Results §1 ¶4 | deutan hV4 adjacent accuracy: produced=0.25  reported=0.25
[OK ] 01.33 Results §1 ¶4 | deutan Crawford-Howell t: produced=-1.889  reported=-1.89
[OK ] 01.34 Results §1 ¶4 | deutan p (one-tailed): produced=0.05388  reported=0.054
[OK ] 01.35 Results §1 ¶4 | deutan d_cc: produced=-2.02  reported=-2.02
[OK ] 01.36 Results §1 ¶4 | protan hV4 adjacent accuracy: produced=0.125  reported=0.13
[OK ] 01.37 Results §1 ¶4 | protan Crawford-Howell t: produced=-3.036  reported=-3.04
[~~ ] 01.38 Results §1 ¶4 | protan p (one-tailed): produced=0.01146  reported=0.012
[OK ] 01.39 Results §1 ¶4 | protan d_cc: produced=-3.245  reported=-3.25
[OK ] 01.40 Results §1 ¶4 | control mean: produced=0.456  reported=0.46
[OK ] 01.41 Results §1 ¶4 | all seven controls above the protan participant: produced=7  reported=7
[OK ] 01.42 Results §1 ¶4 | at least five controls above the deutan parti

### Per-hue vulnerability profile (Results section 1 ¶5; Figure 4C; Supplementary tab:effect_sizes)
Per-hue adjacent accuracy at hV4; one-tailed, uncorrected and exploratory single-case tests.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 01.43 | Results §1 ¶5 | deutan blue adjacent accuracy = 0 | `0.0` |
| 01.44 | Results §1 ¶5 | deutan purple = 0 | `0.0` |
| 01.45 | Results §1 ¶5 | deutan magenta = 0 | `0.0` |
| 01.46 | Results §1 ¶5 | protan blue = 0 | `0.0` |
| 01.47 | Results §1 ¶5 | protan purple = 0 | `0.0` |
| 01.48 | Results §1 ¶5 | protan magenta = 0 | `0.0` |
| 01.49 | tab:effect_sizes | blue t | `-1.93` |
| 01.50 | tab:effect_sizes | blue p | `0.051` |
| 01.51 | tab:effect_sizes | blue d_cc | `-2.06` |
| 01.52 | tab:effect_sizes | purple t | `-0.79` |
| 01.53 | tab:effect_sizes | purple p | `0.229` |
| 01.54 | tab:effect_sizes | purple d_cc | `-0.85` |
| 01.55 | tab:effect_sizes | magenta t | `-1.47` |
| 01.56 | tab:effect_sizes | magenta p | `0.096` |
| 01.57 | tab:effect_sizes | magenta d_cc | `-1.57` |
| 01.58 | Results §1 ¶5 | smallest per-hue p over 16 tests (p >= 0.051 at every hue) | `0.051` |

In [7]:
hc_hue = np.array([adj[s]["hV4"]["adjacent_per_hue"] for s in HC])
per_hue = {}
for name, s in CVD.items():
    v = np.array(adj[s]["hV4"]["adjacent_per_hue"])
    for c, hue in enumerate(HUES):
        per_hue[(name, hue)] = (v[c],) + crawford_howell(v[c], hc_hue[:, c], tail="lower")
for hue in ("blue", "purple", "magenta"):
    print(hue, {n: tuple(round(x, 3) for x in per_hue[(n, hue)]) for n in CVD})
min_p_hue = min(v[2] for v in per_hue.values())
V.check('01.43', 'Results §1 ¶5 | deutan blue adjacent accuracy = 0', per_hue[("deutan","blue")][0], 0.0, mode='eq')
V.check('01.44', 'Results §1 ¶5 | deutan purple = 0', per_hue[("deutan","purple")][0], 0.0, mode='eq')
V.check('01.45', 'Results §1 ¶5 | deutan magenta = 0', per_hue[("deutan","magenta")][0], 0.0, mode='eq')
V.check('01.46', 'Results §1 ¶5 | protan blue = 0', per_hue[("protan","blue")][0], 0.0, mode='eq')
V.check('01.47', 'Results §1 ¶5 | protan purple = 0', per_hue[("protan","purple")][0], 0.0, mode='eq')
V.check('01.48', 'Results §1 ¶5 | protan magenta = 0', per_hue[("protan","magenta")][0], 0.0, mode='eq')
V.check('01.49', 'tab:effect_sizes | blue t', per_hue[("deutan","blue")][1], -1.93, nd=2)
V.check('01.50', 'tab:effect_sizes | blue p', per_hue[("deutan","blue")][2], 0.051, nd=3)
V.check('01.51', 'tab:effect_sizes | blue d_cc', per_hue[("deutan","blue")][3], -2.06, nd=2)
V.check('01.52', 'tab:effect_sizes | purple t', per_hue[("deutan","purple")][1], -0.79, nd=2)
V.check('01.53', 'tab:effect_sizes | purple p', per_hue[("deutan","purple")][2], 0.229, nd=3)
V.check('01.54', 'tab:effect_sizes | purple d_cc', per_hue[("deutan","purple")][3], -0.85, nd=2)
V.check('01.55', 'tab:effect_sizes | magenta t', per_hue[("deutan","magenta")][1], -1.47, nd=2)
V.check('01.56', 'tab:effect_sizes | magenta p', per_hue[("deutan","magenta")][2], 0.096, nd=3)
V.check('01.57', 'tab:effect_sizes | magenta d_cc', per_hue[("deutan","magenta")][3], -1.57, nd=2)
V.check('01.58', 'Results §1 ¶5 | smallest per-hue p over 16 tests (p >= 0.051 at every hue)', min_p_hue, 0.051, nd=3)

blue {'deutan': (0.0, -1.927, 0.051, -2.061), 'protan': (0.0, -1.927, 0.051, -2.061)}
purple {'deutan': (0.0, -0.792, 0.229, -0.847), 'protan': (0.0, -0.792, 0.229, -0.847)}
magenta {'deutan': (0.0, -1.472, 0.096, -1.574), 'protan': (0.0, -1.472, 0.096, -1.574)}
[OK ] 01.43 Results §1 ¶5 | deutan blue adjacent accuracy = 0: produced=0  reported=0
[OK ] 01.44 Results §1 ¶5 | deutan purple = 0: produced=0  reported=0
[OK ] 01.45 Results §1 ¶5 | deutan magenta = 0: produced=0  reported=0
[OK ] 01.46 Results §1 ¶5 | protan blue = 0: produced=0  reported=0
[OK ] 01.47 Results §1 ¶5 | protan purple = 0: produced=0  reported=0
[OK ] 01.48 Results §1 ¶5 | protan magenta = 0: produced=0  reported=0
[OK ] 01.49 tab:effect_sizes | blue t: produced=-1.927  reported=-1.93
[OK ] 01.50 tab:effect_sizes | blue p: produced=0.0511  reported=0.051
[OK ] 01.51 tab:effect_sizes | blue d_cc: produced=-2.061  reported=-2.06
[OK ] 01.52 tab:effect_sizes | purple t: produced=-0.7921  reported=-0.79
[OK ] 01.53

### Interpolation in the Procrustes space, all ROIs (Supplementary tab:alignment)

| id | manuscript | quantity | reported |
|---|---|---|---|
| 01.59 | tab:alignment | LOCO interpolation, Procrustes space: 4 ROIs x (controls, deutan, protan) | `12 cells, see 01.T2.*` |

In [8]:
REP = {"V1": (0.393, 0.437, 0.188), "V2": (0.357, 0.271, 0.104), "V3": (0.339, 0.375, 0.208), "hV4": (0.456, 0.250, 0.125)}
for roi, (hc_r, d_r, p_r) in REP.items():
    V.check(f"01.T2.{roi}.controls", f"tab:alignment LOCO controls {roi}", np.mean([adj[s][roi]["adjacent_mean"] for s in HC]), hc_r, nd=3)
    V.check(f"01.T2.{roi}.deutan", f"tab:alignment LOCO deutan {roi}", adj["sub-08"][roi]["adjacent_mean"], d_r, nd=3)
    V.check(f"01.T2.{roi}.protan", f"tab:alignment LOCO protan {roi}", adj["sub-09"][roi]["adjacent_mean"], p_r, nd=3)
V.table('01.59', 'tab:alignment | LOCO interpolation, Procrustes space: 4 ROIs x (controls, deutan, protan)', '12 cells, see 01.T2.*')

[OK ] 01.T2.V1.controls tab:alignment LOCO controls V1: produced=0.3929  reported=0.393
[OK ] 01.T2.V1.deutan tab:alignment LOCO deutan V1: produced=0.4375  reported=0.437
[OK ] 01.T2.V1.protan tab:alignment LOCO protan V1: produced=0.1875  reported=0.188
[OK ] 01.T2.V2.controls tab:alignment LOCO controls V2: produced=0.3571  reported=0.357
[OK ] 01.T2.V2.deutan tab:alignment LOCO deutan V2: produced=0.2708  reported=0.271
[OK ] 01.T2.V2.protan tab:alignment LOCO protan V2: produced=0.1042  reported=0.104
[OK ] 01.T2.V3.controls tab:alignment LOCO controls V3: produced=0.3393  reported=0.339
[OK ] 01.T2.V3.deutan tab:alignment LOCO deutan V3: produced=0.375  reported=0.375
[OK ] 01.T2.V3.protan tab:alignment LOCO protan V3: produced=0.2083  reported=0.208
[OK ] 01.T2.hV4.controls tab:alignment LOCO controls hV4: produced=0.456  reported=0.456
[OK ] 01.T2.hV4.deutan tab:alignment LOCO deutan hV4: produced=0.25  reported=0.25
[OK ] 01.T2.hV4.protan tab:alignment LOCO protan hV4: produce

### Six decoders under LORO, SRM space (Supplementary S5, tab:loro_decoders)
Control mean ± SD (n = 7) and the two CVD participants. Chance = 0.125.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 01.60 | tab:loro_decoders | LORO six decoders x 4 ROIs: control mean, SD, deutan, protan | `96 cells, see 01.T3.*` |
| 01.61 | S5 ¶2 | LDA has the highest control mean at every ROI | `True` |
| 01.62 | S5 ¶2 | four of the six decoders exceed chance at every ROI (by control mean > 0.125 the count is five, kernel ridge at V3 being 0.158 ± 0.052; by control mean minus one SD it is three: the printed 'four' matches neither criterion) | `4` |

In [9]:
T_LORO = {
 "V1": {"LDA": (0.878, 0.050, 1.000, 0.854), "SVM": (0.777, 0.062, 0.979, 0.771), "ForwardEncoding": (0.542, 0.106, 0.604, 0.625), "Ridge": (0.399, 0.054, 0.354, 0.250), "KernelRidge": (0.393, 0.064, 0.271, 0.292), "MLP": (0.128, 0.008, 0.125, 0.125)},
 "V2": {"LDA": (0.830, 0.047, 0.917, 0.854), "SVM": (0.759, 0.136, 0.875, 0.833), "ForwardEncoding": (0.545, 0.077, 0.458, 0.438), "Ridge": (0.372, 0.097, 0.458, 0.250), "KernelRidge": (0.330, 0.066, 0.313, 0.250), "MLP": (0.149, 0.031, 0.125, 0.125)},
 "V3": {"LDA": (0.726, 0.064, 0.750, 0.771), "SVM": (0.631, 0.144, 0.813, 0.667), "ForwardEncoding": (0.449, 0.065, 0.375, 0.313), "Ridge": (0.173, 0.068, 0.292, 0.229), "KernelRidge": (0.158, 0.052, 0.208, 0.333), "MLP": (0.122, 0.008, 0.125, 0.125)},
 "hV4": {"LDA": (0.658, 0.092, 0.813, 0.729), "SVM": (0.598, 0.085, 0.813, 0.667), "ForwardEncoding": (0.423, 0.072, 0.354, 0.354), "Ridge": (0.319, 0.114, 0.146, 0.250), "KernelRidge": (0.277, 0.093, 0.042, 0.188), "MLP": (0.125, 0.000, 0.125, 0.125)},
}
for roi in ROIS:
    for dec, (m_r, sd_r, d_r, p_r) in T_LORO[roi].items():
        hc = np.array([loro("srm", s, roi, dec) for s in HC])
        V.check(f"01.T3.{roi}.{dec}.mean", f"tab:loro_decoders {roi} {DEC_NAME[dec]} control mean", hc.mean(), m_r, nd=3)
        V.check(f"01.T3.{roi}.{dec}.sd", f"tab:loro_decoders {roi} {DEC_NAME[dec]} control SD", hc.std(ddof=1), sd_r, nd=3)
        V.check(f"01.T3.{roi}.{dec}.deutan", f"tab:loro_decoders {roi} {DEC_NAME[dec]} deutan", loro("srm", "sub-08", roi, dec), d_r, nd=3)
        V.check(f"01.T3.{roi}.{dec}.protan", f"tab:loro_decoders {roi} {DEC_NAME[dec]} protan", loro("srm", "sub-09", roi, dec), p_r, nd=3)
lda_best = all(max(T_LORO[r], key=lambda d: np.mean([loro("srm", s, r, d) for s in HC])) == "LDA" for r in ROIS)
n_above_mean = sum(all(np.mean([loro("srm", s, r, d) for s in HC]) > 0.125 for r in ROIS) for d in DEC)
n_above = sum(all(np.mean([loro("srm", s, r, d) for s in HC]) - np.std([loro("srm", s, r, d) for s in HC], ddof=1) > 0.125 for r in ROIS) for d in DEC)
print("decoders above chance at every ROI: by control mean", n_above_mean, "; by control mean minus one SD", n_above)
V.table('01.60', 'tab:loro_decoders | LORO six decoders x 4 ROIs: control mean, SD, deutan, protan', '96 cells, see 01.T3.*')
V.check('01.61', 'S5 ¶2 | LDA has the highest control mean at every ROI', lda_best, True, mode='eq')
V.check('01.62', "S5 ¶2 | four of the six decoders exceed chance at every ROI (by control mean > 0.125 the count is five, kernel ridge at V3 being 0.158 ± 0.052; by control mean minus one SD it is three: the printed 'four' matches neither criterion)", n_above_mean, 4, mode='eq')

[OK ] 01.T3.V1.LDA.mean tab:loro_decoders V1 LDA control mean: produced=0.878  reported=0.878
[OK ] 01.T3.V1.LDA.sd tab:loro_decoders V1 LDA control SD: produced=0.05021  reported=0.05
[OK ] 01.T3.V1.LDA.deutan tab:loro_decoders V1 LDA deutan: produced=1  reported=1
[OK ] 01.T3.V1.LDA.protan tab:loro_decoders V1 LDA protan: produced=0.8542  reported=0.854
[OK ] 01.T3.V1.SVM.mean tab:loro_decoders V1 SVM control mean: produced=0.7768  reported=0.777
[OK ] 01.T3.V1.SVM.sd tab:loro_decoders V1 SVM control SD: produced=0.06217  reported=0.062
[OK ] 01.T3.V1.SVM.deutan tab:loro_decoders V1 SVM deutan: produced=0.9792  reported=0.979
[OK ] 01.T3.V1.SVM.protan tab:loro_decoders V1 SVM protan: produced=0.7708  reported=0.771
[OK ] 01.T3.V1.ForwardEncoding.mean tab:loro_decoders V1 Hue-channel basis control mean: produced=0.5417  reported=0.542
[OK ] 01.T3.V1.ForwardEncoding.sd tab:loro_decoders V1 Hue-channel basis control SD: produced=0.1055  reported=0.106
[OK ] 01.T3.V1.ForwardEncoding.deut

### Six decoders under LOCO, SRM space (Supplementary S5, tab:loco_decoders)
Adjacent accuracy. Chance is 0.25 for continuous-output decoders and 0.375 for the three classifiers.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 01.63 | tab:loco_decoders | LOCO six decoders x 4 ROIs: control mean, SD, deutan, protan | `96 cells, see 01.T4.*` |
| 01.64 | S5 ¶3 | hue-channel basis exceeds 0.25 at hV4, V1 and V2 (widest margin at hV4) | `True` |
| 01.65 | S5 ¶3 | classifiers remain at or below their 0.375 chance level at every ROI | `0.375` |
| 01.66 | S5 ¶4 | kernel ridge returns exactly 0.000 in every participant and ROI | `0.0` |

In [10]:
T_LOCO = {
 "V1": {"ForwardEncoding": (0.360, 0.098, 0.313, 0.250), "LDA": (0.268, 0.074, 0.375, 0.167), "SVM": (0.226, 0.078, 0.375, 0.229), "Ridge": (0.039, 0.053, 0.146, 0.000), "MLP": (0.375, 0.012, 0.375, 0.375), "KernelRidge": (0.000, 0.000, 0.000, 0.000)},
 "V2": {"ForwardEncoding": (0.283, 0.101, 0.333, 0.063), "LDA": (0.327, 0.088, 0.313, 0.417), "SVM": (0.318, 0.108, 0.250, 0.396), "Ridge": (0.054, 0.083, 0.021, 0.000), "MLP": (0.375, 0.000, 0.396, 0.375), "KernelRidge": (0.000, 0.000, 0.000, 0.000)},
 "V3": {"ForwardEncoding": (0.220, 0.100, 0.354, 0.125), "LDA": (0.220, 0.081, 0.271, 0.333), "SVM": (0.208, 0.103, 0.250, 0.229), "Ridge": (0.000, 0.000, 0.000, 0.000), "MLP": (0.250, 0.000, 0.250, 0.250), "KernelRidge": (0.000, 0.000, 0.000, 0.000)},
 "hV4": {"ForwardEncoding": (0.470, 0.130, 0.271, 0.104), "LDA": (0.280, 0.144, 0.146, 0.229), "SVM": (0.253, 0.126, 0.167, 0.208), "Ridge": (0.000, 0.000, 0.000, 0.000), "MLP": (0.250, 0.000, 0.250, 0.250), "KernelRidge": (0.000, 0.000, 0.000, 0.000)},
}
for roi in ROIS:
    for dec, (m_r, sd_r, d_r, p_r) in T_LOCO[roi].items():
        hc = np.array([loco_srm(s, roi, dec) for s in HC])
        V.check(f"01.T4.{roi}.{dec}.mean", f"tab:loco_decoders {roi} {DEC_NAME[dec]} control mean", hc.mean(), m_r, nd=3)
        V.check(f"01.T4.{roi}.{dec}.sd", f"tab:loco_decoders {roi} {DEC_NAME[dec]} control SD", hc.std(ddof=1), sd_r, nd=3)
        V.check(f"01.T4.{roi}.{dec}.deutan", f"tab:loco_decoders {roi} {DEC_NAME[dec]} deutan", loco_srm("sub-08", roi, dec), d_r, nd=3)
        V.check(f"01.T4.{roi}.{dec}.protan", f"tab:loco_decoders {roi} {DEC_NAME[dec]} protan", loco_srm("sub-09", roi, dec), p_r, nd=3)
fe_hc = {r: np.mean([loco_srm(s, r, "ForwardEncoding") for s in HC]) for r in ROIS}
cls_max = max(np.mean([loco_srm(s, r, d) for s in HC]) for r in ROIS for d in ("LDA", "SVM", "MLP"))
kr_max = max(loco_srm(s, r, "KernelRidge") for s in SUBS for r in ROIS)
V.table('01.63', 'tab:loco_decoders | LOCO six decoders x 4 ROIs: control mean, SD, deutan, protan', '96 cells, see 01.T4.*')
V.check('01.64', 'S5 ¶3 | hue-channel basis exceeds 0.25 at hV4, V1 and V2 (widest margin at hV4)', fe_hc["hV4"] - 0.25 > max(fe_hc["V1"], fe_hc["V2"]) - 0.25 and min(fe_hc["V1"], fe_hc["V2"]) > 0.25, True, mode='eq')
V.check('01.65', 'S5 ¶3 | classifiers remain at or below their 0.375 chance level at every ROI', cls_max, 0.375, mode='le')
V.check('01.66', 'S5 ¶4 | kernel ridge returns exactly 0.000 in every participant and ROI', kr_max, 0.0, mode='eq')

[OK ] 01.T4.V1.ForwardEncoding.mean tab:loco_decoders V1 Hue-channel basis control mean: produced=0.3601  reported=0.36
[OK ] 01.T4.V1.ForwardEncoding.sd tab:loco_decoders V1 Hue-channel basis control SD: produced=0.09824  reported=0.098
[OK ] 01.T4.V1.ForwardEncoding.deutan tab:loco_decoders V1 Hue-channel basis deutan: produced=0.3125  reported=0.313
[OK ] 01.T4.V1.ForwardEncoding.protan tab:loco_decoders V1 Hue-channel basis protan: produced=0.25  reported=0.25
[OK ] 01.T4.V1.LDA.mean tab:loco_decoders V1 LDA control mean: produced=0.2679  reported=0.268
[OK ] 01.T4.V1.LDA.sd tab:loco_decoders V1 LDA control SD: produced=0.07359  reported=0.074
[OK ] 01.T4.V1.LDA.deutan tab:loco_decoders V1 LDA deutan: produced=0.375  reported=0.375
[OK ] 01.T4.V1.LDA.protan tab:loco_decoders V1 LDA protan: produced=0.1667  reported=0.167
[OK ] 01.T4.V1.SVM.mean tab:loco_decoders V1 SVM control mean: produced=0.2262  reported=0.226
[OK ] 01.T4.V1.SVM.sd tab:loco_decoders V1 SVM control SD: produced=

### Leakage bound of the fixed-reference alignment (Supplementary S7)
Re-estimating the run alignment inside each fold (`nested_only`) versus the fixed run-1 reference (`raw_ctrl`), hue-channel basis LORO accuracy.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 01.67 | S7 | fixed-reference hue-channel LORO accuracy (pooled over ten participants incl. sub-10, as printed) | `0.545` |
| 01.68 | S7 | nested re-estimation raises it to (same pool) | `0.578` |
| 01.68b | S7 | the increase persists with sub-10 excluded (nine participants) | `True` |

In [11]:
def loro_nested(sub, roi):
    j = J(f"nested_procrustes/nested_only/{sub}_performance_raw.json")["results"]
    space = list(j.keys())[0]
    return float(np.mean([f["acc_exact"] for f in j[space][RDIR[roi]]["ForwardEncoding"]]))
# The printed S7 values pool ten participants (V1-hV4), i.e. they include sub-10, whom the
# manuscript excludes elsewhere. Both the ten- and the nine-participant pools are shown.
SUBS10 = SUBS + ["sub-10"]
fixed10 = np.mean([loro("procrustes", s, r) for s in SUBS10 for r in ROIS]); nested10 = np.mean([loro_nested(s, r) for s in SUBS10 for r in ROIS])
fixed9 = np.mean([loro("procrustes", s, r) for s in SUBS for r in ROIS]); nested9 = np.mean([loro_nested(s, r) for s in SUBS for r in ROIS])
print(f"ten participants (as printed): fixed run-1 {fixed10:.4f} -> nested {nested10:.4f}; nine participants (sub-10 excluded): {fixed9:.4f} -> {nested9:.4f}")
V.check('01.67', 'S7 | fixed-reference hue-channel LORO accuracy (pooled over ten participants incl. sub-10, as printed)', fixed10, 0.545, nd=3)
V.check('01.68', 'S7 | nested re-estimation raises it to (same pool)', nested10, 0.578, nd=3)
V.check('01.68b', 'S7 | the increase persists with sub-10 excluded (nine participants)', nested9 > fixed9, True, mode='eq')

ten participants (as printed): fixed run-1 0.5448 -> nested 0.5781; nine participants (sub-10 excluded): 0.5434 -> 0.5666
[OK ] 01.67 S7 | fixed-reference hue-channel LORO accuracy (pooled over ten participants incl. sub-10, as printed): produced=0.5448  reported=0.545
[OK ] 01.68 S7 | nested re-estimation raises it to (same pool): produced=0.5781  reported=0.578
[OK ] 01.68b S7 | the increase persists with sub-10 excluded (nine participants): produced=True  reported=True


### Alignment robustness of the within-participant readouts (Supplementary S17)

| id | manuscript | quantity | reported |
|---|---|---|---|
| 01.69 | S17 | lowest CVD classification cell over both spaces | `0.312` |
| 01.70 | S17 | Procrustes gives the higher control classification mean at all four ROIs | `4` |
| 01.71 | S17 | and the higher control interpolation mean at three of four | `3` |
| 01.72 | S17 | Procrustes higher in 26 of 36 participant-by-region classification cells | `26` |
| 01.73 | S17 | SRM V1 classification d_cc, deutan | `0.59` |
| 01.74 | S17 | SRM V1 classification d_cc, protan | `0.79` |
| 01.75 | S17 | Procrustes V1 classification d_cc, both | `-0.25` |

In [12]:
def ch_space(space, s, roi):
    hc = [LORO_P[space][h][roi] for h in HC]
    return crawford_howell(LORO_P[space][s][roi], hc, tail="two")[2]
srm_v1 = (ch_space("srm", "sub-08", "V1"), ch_space("srm", "sub-09", "V1"))
pro_v1 = (ch_space("procrustes", "sub-08", "V1"), ch_space("procrustes", "sub-09", "V1"))
lowest_both = min(LORO_P[sp][s][r] for sp in ("procrustes", "srm") for s in ("sub-08", "sub-09") for r in ROIS)
loro_ctrl_higher = sum(np.mean([LORO_P["procrustes"][s][r] for s in HC]) > np.mean([LORO_P["srm"][s][r] for s in HC]) for r in ROIS)
loco_ctrl_higher = sum(np.mean([adj[s][r]["adjacent_mean"] for s in HC]) > np.mean([loco_srm(s, r) for s in HC]) for r in ROIS)
cells_higher = sum(LORO_P["procrustes"][s][r] > LORO_P["srm"][s][r] for s in SUBS for r in ROIS)
print(srm_v1, pro_v1, lowest_both, loro_ctrl_higher, loco_ctrl_higher, cells_higher)
V.check('01.69', 'S17 | lowest CVD classification cell over both spaces', lowest_both, 0.312, nd=3)
V.check('01.70', 'S17 | Procrustes gives the higher control classification mean at all four ROIs', loro_ctrl_higher, 4, mode='eq')
V.check('01.71', 'S17 | and the higher control interpolation mean at three of four', loco_ctrl_higher, 3, mode='eq')
V.check('01.72', 'S17 | Procrustes higher in 26 of 36 participant-by-region classification cells', cells_higher, 26, mode='eq')
V.check('01.73', 'S17 | SRM V1 classification d_cc, deutan', srm_v1[0], 0.59, nd=2)
V.check('01.74', 'S17 | SRM V1 classification d_cc, protan', srm_v1[1], 0.79, nd=2)
V.check('01.75', 'S17 | Procrustes V1 classification d_cc, both', pro_v1[0], -0.25, nd=2)

(0.592156525463792, 0.7895420339517231) (-0.25301969999050333, -0.25301969999050333) 0.3125 4 3 26
[OK ] 01.69 S17 | lowest CVD classification cell over both spaces: produced=0.3125  reported=0.312
[OK ] 01.70 S17 | Procrustes gives the higher control classification mean at all four ROIs: produced=4  reported=4
[OK ] 01.71 S17 | and the higher control interpolation mean at three of four: produced=3  reported=3
[OK ] 01.72 S17 | Procrustes higher in 26 of 36 participant-by-region classification cells: produced=26  reported=26
[OK ] 01.73 S17 | SRM V1 classification d_cc, deutan: produced=0.5922  reported=0.59
[OK ] 01.74 S17 | SRM V1 classification d_cc, protan: produced=0.7895  reported=0.79
[OK ] 01.75 S17 | Procrustes V1 classification d_cc, both: produced=-0.253  reported=-0.25


### Ridge-penalty grid (Supplementary S6)

| id | manuscript | quantity | reported |
|---|---|---|---|
| 01.76 | S6 | GCV searches seven penalties from 1e-3 to 1e3 | `(7, 0.001, 1000.0)` |

In [13]:
ls = J("lambda_stability.json"); grid = ls["alpha_grid"]
print(grid)
V.check('01.76', 'S6 | GCV searches seven penalties from 1e-3 to 1e3', (len(grid), min(grid), max(grid)), (7, 0.001, 1000.0), mode='eq')

[0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
[OK ] 01.76 S6 | GCV searches seven penalties from 1e-3 to 1e3: produced=(7, 0.001, 1000)  reported=(7, 0.001, 1000)


In [14]:
V.summary()


=== 01_decoding: 285/289 numeric checks reproduced exactly; 3 within one unit of the last printed digit; 1 mismatch, 0 error, 0 pointer-only ===
  NEAR     01.38 Results §1 ¶4 | protan p (one-tailed): produced=0.01146 reported=0.012
  NEAR     01.T3.V2.MLP.sd tab:loro_decoders V2 MLP control SD: produced=0.0305 reported=0.031
  NEAR     01.T3.hV4.Ridge.mean tab:loro_decoders hV4 Ridge control mean: produced=0.3185 reported=0.319
  MISMATCH 01.62 S5 ¶2 | four of the six decoders exceed chance at every ROI (by control mean > 0.125 the count is five, kernel ridge at V3 being 0.158 ± 0.052; by control mean minus one SD it is three: the printed 'four' matches neither criterion): produced=5 reported=4
